In [334]:
import torch
import pandas as pd
import numpy as np
import math
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from transformers import AutoModel

In [340]:
dn = 768
dk = 128
dv = 128
b = 100
p = 128
distilbert = AutoModel.from_pretrained("distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
vocabI = tokenizer.vocab_size
mp = distilbert.embeddings.word_embeddings.weight.detach().clone().requires_grad_(False)
W_K = (torch.randn(dn,dk)/math.sqrt(dn)).requires_grad_(True)
W_Q = (torch.randn(dn,dk)/math.sqrt(dn)).requires_grad_(True)
W_V = (torch.randn(dn,dv)/math.sqrt(dn)).requires_grad_(True)
W = (torch.randn(dv,p)/math.sqrt(dv)).requires_grad_(True)
B = torch.zeros(p,requires_grad=True)
W1 = (torch.randn(p,1)/math.sqrt(p)).requires_grad_(True)
B1 = torch.zeros(1,requires_grad=True)
with torch.no_grad():
    mp[0] = 0

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [341]:
d1 = pd.read_csv("train.csv")
text = d1["text"]
target = d1["target"]
train_idx, val_idx = train_test_split(range(len(text)), test_size=0.2, random_state=42)

train_text   = text.iloc[train_idx].reset_index(drop=True)
val_text     = text.iloc[val_idx].reset_index(drop=True)
train_target = target.iloc[train_idx].reset_index(drop=True)
val_target   = target.iloc[val_idx].reset_index(drop=True)

In [342]:
adam = torch.optim.Adam([W_K,W_Q,W_V,W,W1,B,B1],lr=1e-3,weight_decay=0.01)

In [343]:
def batch(textsorc, targetsorc,batch_size):
    pl = torch.randint(0,len(textsorc),(batch_size,)).tolist()
    lola = textsorc[pl].tolist()
    targ = targetsorc[pl]

    enc = tokenizer(lola,padding='max_length', truncation=True, max_legth=62, return_tensors='pt')
    ids = enc['input_ids']
    ba = mp[ids]
    fr = enc['attention_mask'].bool()
    papa = torch.tensor(targ.values,dtype=torch.float32)
    
    return ba,fr,papa
def net(X, mask,is_training):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    scores = Q @ K.transpose(-2,-1)/math.sqrt(dk)
    key_mask = mask.unsqueeze(1)
    scores = scores.masked_fill(~key_mask,float('-inf'))
    alph = torch.softmax(scores,dim=-1)
    Y = alph @ V
    #Y = Y.mean(dim=1)
    mask1 = mask.unsqueeze(-1)
    Y = (Y * mask1).sum(dim=1)/mask1.sum(dim=1)
    Y = torch.nn.functional.dropout(Y, p=0.3,training=is_training)
    Y_norm = (Y - Y.mean(-1,keepdim=True))/(Y.std(-1,keepdim=True)+1e-5)
    h = torch.tanh(Y_norm @ W + B)
    h = torch.nn.functional.dropout(h,p=0.3,training=is_training)
    predict = h @ W1 + B1
    predict = predict.squeeze(-1)
    return predict

In [344]:

for step in range(2000):
    X,mask,pl = batch(train_text,train_target,b)
    predict = net(X,mask,True)
    #pass into MLP
    
    loss = torch.nn.functional.binary_cross_entropy_with_logits(predict,pl)
    if step % 20 == 0: 
        with torch.no_grad():
            X_v,mask_v,pl_v = batch(val_text,val_target,len(val_text))
            predict1 = net(X_v,mask_v,False)
            loss1 = torch.nn.functional.binary_cross_entropy_with_logits(predict1,pl_v)
            print(step,loss.item(), loss1.item())
    adam.zero_grad()
    loss.backward()
       
    adam.step()

0 0.7747418880462646 0.8532724976539612
20 0.775519609451294 0.6932419538497925
40 0.7016451954841614 0.6407449245452881
60 0.6136414408683777 0.6394945383071899
80 0.6819635033607483 0.6067756414413452
100 0.6180579662322998 0.605803906917572
120 0.6321485042572021 0.5955235958099365
140 0.6067473888397217 0.6122703552246094
160 0.5351555347442627 0.5509774088859558
180 0.5396678447723389 0.5254858136177063
200 0.4914412796497345 0.5346889495849609
220 0.5222686529159546 0.5278368592262268
240 0.7321807742118835 0.6275472640991211
260 0.5149402618408203 0.5256327390670776
280 0.5103890299797058 0.5267692804336548
300 0.6277835369110107 0.6120701432228088
320 0.605808436870575 0.5247268676757812
340 0.6359584927558899 0.5922213792800903
360 0.5660496354103088 0.5306575894355774
380 0.5929273366928101 0.5495067834854126
400 0.48728039860725403 0.586499810218811
420 0.4796198904514313 0.505183756351471
440 0.4541376829147339 0.4857804477214813
460 0.5146259069442749 0.5406325459480286
48

KeyboardInterrupt: 

In [240]:
a = input()
a

 adda


'adda'

In [282]:
def create(b:int,phrase:str):
    
    lola = [phrase]
  
    enc = tokenizer(lola,padding='max_length', truncation=True, max_legth=62, return_tensors='pt')
    ids = enc['input_ids']
    ba = mp[ids]
    fr = enc['attention_mask'].bool()
    return ba,fr


In [358]:
d2 = pd.read_csv("test.csv")
text1 = d2["text"]
ids = d2["id"]


lol = []
for i,pos in zip(text1,ids):
    ba,fr = create(1,i)
    with torch.no_grad():
        predict = net(ba,fr,False)
        lol.append(0 if torch.sigmoid(predict).item() < 0.5 else 1)
out = pd.DataFrame({"id":ids,"target":lol})
out.to_csv("submission.csv",index=False)

In [349]:

ph1 = "Accident center lane blocked in #SantaClara on US-101 NB before Great America Pkwy #BayArea #Traffic http://t.co/pmlOhZuRWR"
ph2 = "I can't have kids cuz I got in a bicycle accident &amp; split my testicles. it's impossible for me to have kids' MICHAEL YOU ARE THE FATHER"
ph3 = "No I don't like cold!"
ph4 = "A girl who died in an airplane accident fifteen years ago"
ph5 = "Horrible Accident  Man Died In Wings of AirplaneåÊ(29-07-2015) http://t.co/5ZRKZdhODe"
ba,fr = create(1,ph1)
with torch.no_grad():
    predict = net(ba,fr,False)
    print(torch.sigmoid(predict))

tensor([0.9553])


30522